In [1]:
import numpy as np
import scipy.signal as sig
from scipy.io import wavfile


# ---------------------------
# Load + normalize audio
# ---------------------------
def load_wav_mono(path):
    sr, x = wavfile.read(path)
    x = x.astype(float)

    if x.ndim == 2:
        x = x.mean(axis=1)

    x = x - np.mean(x)
    peak = np.max(np.abs(x))
    if peak > 0:
        x = x / peak

    return sr, x


# ---------------------------
# Select steady-state region
# ---------------------------
def select_steady_state_segment(x, sr, choose_sec=6.0):
    frame_len = int(1.0 * sr)
    hop_len = int(0.25 * sr)
    choose_len = int(choose_sec * sr)

    if len(x) <= choose_len:
        return x

    starts = np.arange(0, len(x) - frame_len, hop_len)

    rms_vals = []
    centroid_vals = []

    for s in starts:
        frame = x[s:s + frame_len]
        rms = np.sqrt(np.mean(frame**2))

        freqs, psd = sig.welch(frame, fs=sr, nperseg=min(4096, len(frame)))
        centroid = np.sum(freqs * psd) / np.sum(psd) if np.sum(psd) > 0 else 0

        rms_vals.append(rms)
        centroid_vals.append(centroid)

    rms_vals = np.array(rms_vals)
    centroid_vals = np.array(centroid_vals)

    best_score = np.inf
    best_start = 0

    window = int(choose_sec / 0.25)

    for i in range(len(starts) - window):
        score = np.std(rms_vals[i:i+window]) + 0.001 * np.std(centroid_vals[i:i+window])
        if score < best_score:
            best_score = score
            best_start = starts[i]

    return x[best_start:best_start + choose_len]


# ---------------------------
# Bandpass filter
# ---------------------------
def bandpass_filter(x, sr, low=20, high=1200):
    sos = sig.butter(6, [low, high], btype="bandpass", fs=sr, output="sos")
    return sig.sosfiltfilt(sos, x)


# ---------------------------
# Power spectrum
# ---------------------------
def welch_spectrum(x, sr):
    nperseg = min(32768, len(x))
    f, pxx = sig.welch(x, fs=sr, nperseg=nperseg)
    return f, pxx


# ---------------------------
# Harmonic detection (KEY PART)
# ---------------------------
def harmonic_sum_score(freqs, power, fmin=10, fmax=150):
    candidates = np.arange(fmin, fmax, 0.1)
    scores = []

    for f0 in candidates:
        score = 0
        weight_sum = 0

        for k in range(1, 8):
            fk = k * f0
            if fk > freqs[-1]:
                break

            idx = np.argmin(np.abs(freqs - fk))
            w = 1 / k
            score += w * power[idx]
            weight_sum += w

        scores.append(score / weight_sum if weight_sum > 0 else 0)

    scores = np.array(scores)
    best_idx = np.argmax(scores)

    return candidates[best_idx], scores


# ---------------------------
# Cepstrum check (advanced validation)
# ---------------------------
def cepstrum_check(x, sr):
    n = min(len(x), 65536)
    x = x[:n] * np.hanning(n)

    spec = np.abs(np.fft.rfft(x)) + 1e-12
    ceps = np.fft.irfft(np.log(spec))

    quef = np.arange(len(ceps)) / sr

    mask = (quef > 1/150) & (quef < 1/10)

    if np.any(mask):
        q_peak = quef[mask][np.argmax(ceps[mask])]
        return 1 / q_peak

    return None


# ---------------------------
# MAIN FUNCTION
# ---------------------------
def estimate_rpm_from_wav(wav_path, cylinders=4, strokes=4):

    sr, x = load_wav_mono(wav_path)

    # Step 1: steady-state
    x = select_steady_state_segment(x, sr)

    # Step 2: filter
    x = bandpass_filter(x, sr)

    # Step 3: spectrum
    freqs, pxx = welch_spectrum(x, sr)

    mask = (freqs >= 10) & (freqs <= 300)
    freqs = freqs[mask]
    pxx = pxx[mask]

    # Step 4: harmonic detection
    f_detected, scores = harmonic_sum_score(freqs, pxx)

    # Step 5: cepstrum validation
    f_cep = cepstrum_check(x, sr)

    # Step 6: RPM calculations
    rpm_direct = 60 * f_detected

    if strokes == 4:
        rpm_engine = (120 * f_detected) / cylinders
    else:
        rpm_engine = (60 * f_detected) / cylinders

    return {
    "Detected Frequency (Hz)": round(float(f_detected), 2),
    "Cepstrum Check (Hz)": round(float(f_cep), 2) if f_cep is not None else None,
    "RPM (direct)": round(float(rpm_direct), 0),
    "RPM (engine model)": round(float(rpm_engine), 0)
}

In [2]:
result = estimate_rpm_from_wav(
    "ToyotaRav4_3000RPM.wav",
    cylinders=4,
    strokes=4
)

result

{'Detected Frequency (Hz)': 106.3,
 'Cepstrum Check (Hz)': 22.49,
 'RPM (direct)': 6378.0,
 'RPM (engine model)': 3189.0}